# Train SeasonCNN on a GPU

Runner notebook for `CNN-models/train.py` -- use this on a machine/runtime
with an actual GPU (e.g. a Google Colab GPU runtime), since the local dev
machine this repo was built on has neither CUDA nor MPS available.

This notebook does **not** reimplement the training loop -- it just handles
environment setup (Colab: clone repo, mount Drive for `data/processed/`,
install deps) and then shells out to `train.py`, so architecture/hyperparameter
edits in `model.py`/`train.py` are picked up automatically with no need to
keep two copies of the training logic in sync.

**Before running on Colab:** upload `data/processed/` (the
`annotations_processed.csv` + `RGB/` produced by the preprocessing pipeline,
~430MB) once to `MyDrive/ColourAnalysis/data/processed` in Google Drive --
it's too large to keep in git, so this is the hand-off point.

## 1. Environment setup

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)

In [ ]:
def find_repo_root(start: Path) -> Path:
    """Walk upward from `start` looking for the repo root (marked by
    requirements.txt + src/), since a local Jupyter server's cwd isn't
    necessarily this notebook's directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("could not locate repo root (expected requirements.txt + src/)")


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    REPO_URL = "https://github.com/cheryl-bao/Colour-Analysis-using-the-Four-Season-System-via-Deep-Learning.git"
    REPO_DIR = Path("/content/Colour-Analysis-using-the-Four-Season-System-via-Deep-Learning")

    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull

    !pip install -q -r {REPO_DIR}/requirements.txt

    # data/processed/ is too large for git -- symlink it in from Drive
    # instead of copying, so re-running this cell is cheap.
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/ColourAnalysis/data/processed")
    repo_data_dir = REPO_DIR / "data" / "processed"
    if not repo_data_dir.exists():
        if not DRIVE_DATA_DIR.exists():
            raise FileNotFoundError(
                f"expected processed data at {DRIVE_DATA_DIR} -- upload data/processed/ there first"
            )
        repo_data_dir.parent.mkdir(parents=True, exist_ok=True)
        repo_data_dir.symlink_to(DRIVE_DATA_DIR, target_is_directory=True)
else:
    REPO_DIR = find_repo_root(Path.cwd())

sys.path.insert(0, str(REPO_DIR))
print("repo root:", REPO_DIR)

## 2. Confirm a GPU is visible

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    print("MPS (Apple GPU) available")
else:
    print("No GPU detected -- train.py will fall back to CPU")

## 3. Train

Set hyperparameters below and run -- this shells out to `CNN-models/train.py`
so it's the exact same code path as running it from the terminal. `train.py`
auto-detects the GPU (see `get_device()`), so no extra flag is needed here.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-3
VAL_FRAC = 0.15
NUM_WORKERS = 2
SEED = 42

In [ ]:
!cd {REPO_DIR} && python -u CNN-models/train.py \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --val-frac {VAL_FRAC} \
    --num-workers {NUM_WORKERS} \
    --seed {SEED}

## 4. Inspect results

In [ ]:
import json

import matplotlib.pyplot as plt

results_path = REPO_DIR / "CNN-models" / "results.json"
results = json.loads(results_path.read_text())
history = results["history"]
epochs = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(epochs, [h["train_acc"] for h in history], label="train")
axes[1].plot(epochs, [h["val_acc"] for h in history], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()

plt.tight_layout()
plt.show()

print(f"best val accuracy: {results['best_val_acc']:.4f}")
print(f"test accuracy:     {results['accuracy']:.4f}")